# RQ1 — Missing Metrics: BLEU + Exact Match + BERTScore + Statistical Tests

**Requires:** `rq1_responses.csv` and `rq1_metrics.csv` already saved in Google Drive  
**Tip:** Switch runtime to **GPU (T4)** before running — speeds up BERTScore from ~20 min → ~5 min  
*(Runtime → Change runtime type → T4 GPU)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 1 — BLEU + Exact Match  (~2 min, CPU)

In [ ]:
!pip install -q nltk

import pandas as pd
import numpy as np
from itertools import combinations
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Paths ─────────────────────────────────────────────────────────────────────
RESPONSES_PATH    = "/content/drive/MyDrive/Thesis/rq1_responses.csv"
METRICS_PATH      = "/content/drive/MyDrive/Thesis/rq1_metrics.csv"
COMPLETE_OUT_PATH = "/content/drive/MyDrive/Thesis/rq1_metrics_complete.csv"

# ── Load ──────────────────────────────────────────────────────────────────────
df         = pd.read_csv(RESPONSES_PATH)
metrics_df = pd.read_csv(METRICS_PATH)

# Keep 3 final models only (drop older llama-3.3-70b data)
MODELS = ["llama-4-scout", "llama-3.1-8b", "mistral-7b"]
df = df[df["model"].isin(MODELS)].copy()

print(f"Responses loaded : {len(df)}")
print(df.groupby("model")["Q_id"].count().to_string())
print(f"\nExisting metrics : {len(metrics_df)} rows")
print(f"Columns          : {list(metrics_df.columns)}")

# ── Helper ────────────────────────────────────────────────────────────────────
def normalize(text):
    return " ".join(str(text).lower().strip().split())

smoothie = SmoothingFunction().method1
bleu_em_records = []
groups = list(df.groupby(["Q_id", "model"]))
print(f"\nComputing BLEU + Exact Match for {len(groups)} groups...")

for i, ((qid, model), grp) in enumerate(groups):
    responses = grp["response"].dropna().tolist()
    if len(responses) < 2:
        continue

    norm  = [normalize(r) for r in responses]
    pairs = list(combinations(range(len(norm)), 2))

    # Symmetric pairwise BLEU
    bleu_scores = []
    for a, b in pairs:
        ab = sentence_bleu([norm[a].split()], norm[b].split(), smoothing_function=smoothie)
        ba = sentence_bleu([norm[b].split()], norm[a].split(), smoothing_function=smoothie)
        bleu_scores.append((ab + ba) / 2)

    # Exact Match: fraction of pairs character-identical after normalization
    exact_scores = [1.0 if norm[a] == norm[b] else 0.0 for a, b in pairs]

    bleu_em_records.append({
        "Q_id":       qid,
        "model":      model,
        "BLEU":       round(float(np.mean(bleu_scores)), 4),
        "ExactMatch": round(float(np.mean(exact_scores)), 4),
    })

    if (i + 1) % 60 == 0:
        print(f"  [{i+1}/{len(groups)}] done...")

bleu_em_df = pd.DataFrame(bleu_em_records)
print(f"\nDone. {len(bleu_em_df)} groups computed.")
print(bleu_em_df.groupby("model")[["BLEU", "ExactMatch"]].mean().round(4).to_string())

Responses loaded : 1500
model
llama-3.1-8b     500
llama-4-scout    500
mistral-7b       500

Existing metrics : 300 rows
Columns          : ['Q_id', 'Type', 'model', 'SAS', 'ROUGE_L', 'Sem_Var', 'CR', 'n_responses']

Computing BLEU + Exact Match for 300 groups...
  [60/300] done...
  [120/300] done...
  [180/300] done...
  [240/300] done...
  [300/300] done...

Done. 300 groups computed.
                 BLEU  ExactMatch
model                            
llama-3.1-8b   0.1497         0.0
llama-4-scout  0.2894         0.0
mistral-7b     0.1148         0.0


## Cell 2 — BERTScore  (~5 min on GPU · ~20 min on CPU)
Make sure you are using a **GPU runtime** before running this cell.

In [ ]:
!pip install -q bert-score

from bert_score import score as bert_score_fn

bs_records = []
groups_list = list(df.groupby(["Q_id", "model"]))
total = len(groups_list)
print(f"Computing BERTScore for {total} groups (10 pairs each)...")

for i, ((qid, model), grp) in enumerate(groups_list):
    responses = grp["response"].dropna().tolist()
    if len(responses) < 2:
        continue

    pairs = list(combinations(range(len(responses)), 2))
    cands = [responses[b] for a, b in pairs]   # hypotheses
    refs  = [responses[a] for a, b in pairs]   # references

    P, R, F1 = bert_score_fn(cands, refs, lang="en", verbose=False)
    bs_f1 = round(float(F1.mean().item()), 4)

    bs_records.append({"Q_id": qid, "model": model, "BERTScore": bs_f1})

    if (i + 1) % 30 == 0:
        print(f"  [{i+1}/{total}] done...")

bs_df = pd.DataFrame(bs_records)
print(f"\nDone. {len(bs_df)} groups computed.")
print(bs_df.groupby("model")["BERTScore"].mean().round(4).to_string())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
Computing BERTScore for 300 groups (10 pairs each)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (608 outputs left out)

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [300/300] done...

Done. 300 groups computed.
model
llama-3.1-8b     0.8944
llama-4-scout    0.9175
mistral-7b       0.8929


## Cell 3 — Merge + Save complete CSV

In [ ]:
complete = (
    metrics_df
    .merge(bleu_em_df, on=["Q_id", "model"], how="left")
    .merge(bs_df,      on=["Q_id", "model"], how="left")
)

# Enforce column order
col_order = ["Q_id", "Type", "model",
             "SAS", "ROUGE_L", "BLEU", "ExactMatch",
             "BERTScore", "Sem_Var", "CR", "n_responses"]
complete = complete[[c for c in col_order if c in complete.columns]]

complete.to_csv(COMPLETE_OUT_PATH, index=False)
print(f"Saved → {COMPLETE_OUT_PATH}")
print(f"Shape  : {complete.shape}")

print("\n── Mean metrics by MODEL ──")
print(complete.groupby("model")[["SAS","ROUGE_L","BLEU","BERTScore","CR"]].mean().round(4).to_string())

print("\n── Mean metrics by QUESTION TYPE ──")
print(complete.groupby("Type")[["SAS","ROUGE_L","BLEU","BERTScore","CR"]].mean().round(4).to_string())

Saved → /content/drive/MyDrive/Thesis/rq1_metrics_complete.csv
Shape  : (300, 11)

── Mean metrics by MODEL ──
                  SAS  ROUGE_L    BLEU  BERTScore     CR
model                                                   
llama-3.1-8b   0.8788   0.3116  0.1497     0.8944  0.774
llama-4-scout  0.9175   0.4652  0.2894     0.9175  0.916
mistral-7b     0.8799   0.3223  0.1148     0.8929  0.759

── Mean metrics by QUESTION TYPE ──
                 SAS  ROUGE_L    BLEU  BERTScore      CR
Type                                                    
ADVISORY      0.8936   0.3771  0.1820     0.9024  0.8213
FACTUAL       0.8844   0.3693  0.1774     0.8974  0.7813
HYPOTHETICAL  0.8978   0.3695  0.2007     0.9045  0.8467
PROCEDURAL    0.8924   0.3495  0.1783     0.9022  0.8160


## Cell 4 — Statistical Tests: ANOVA + Tukey HSD
Copy the F-stats and p-values directly into your thesis (Chapter 10 / 11).

In [ ]:
!pip install -q statsmodels

from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Reload in case this cell is run fresh
complete = pd.read_csv(COMPLETE_OUT_PATH)

Q_TYPES = ["FACTUAL", "PROCEDURAL", "ADVISORY", "HYPOTHETICAL"]
METRICS = ["SAS", "ROUGE_L", "BLEU", "BERTScore"]

def sig_label(p):
    if p < 0.001: return "*** p<0.001"
    if p < 0.01:  return "**  p<0.01"
    if p < 0.05:  return "*   p<0.05"
    return "ns  (not significant)"

# ── ANOVA by Question Type ───────────────────────────────────────────────────
print("=" * 65)
print("ONE-WAY ANOVA — metrics by QUESTION TYPE")
print("=" * 65)
for metric in METRICS:
    if metric not in complete.columns:
        print(f"{metric}: column not found, skipping")
        continue
    groups = [complete[complete["Type"] == t][metric].dropna().values for t in Q_TYPES]
    f, p   = stats.f_oneway(*groups)
    print(f"\n{metric}: F={f:.4f}, p={p:.6f}  →  {sig_label(p)}")
    if p < 0.05:
        sub = complete[["Type", metric]].dropna()
        tukey = pairwise_tukeyhsd(sub[metric], sub["Type"])
        print(tukey.summary())

# ── ANOVA by Model ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("ONE-WAY ANOVA — metrics by MODEL")
print("=" * 65)
for metric in METRICS:
    if metric not in complete.columns:
        continue
    groups = [complete[complete["model"] == m][metric].dropna().values for m in MODELS]
    f, p   = stats.f_oneway(*groups)
    print(f"\n{metric}: F={f:.4f}, p={p:.6f}  →  {sig_label(p)}")
    if p < 0.05:
        sub = complete[["model", metric]].dropna()
        tukey = pairwise_tukeyhsd(sub[metric], sub["model"])
        print(tukey.summary())

# ── Levene Test — variance homogeneity check ─────────────────────────────────
print("\n" + "=" * 65)
print("LEVENE TEST — Variance homogeneity (SAS by Question Type)")
print("=" * 65)
groups_lev = [complete[complete["Type"] == t]["SAS"].dropna().values for t in Q_TYPES]
lev_stat, lev_p = stats.levene(*groups_lev)
print(f"Levene: W={lev_stat:.4f}, p={lev_p:.6f}")
print("→ Equal variances assumed (ANOVA assumption holds)." if lev_p > 0.05
      else "→ Variances differ — consider Welch ANOVA as robustness check.")

print("\nAll done. Copy F-stats + p-values into your thesis.")

ONE-WAY ANOVA — metrics by QUESTION TYPE

SAS: F=1.1750, p=0.319475  →  ns  (not significant)

ROUGE_L: F=0.9277, p=0.427686  →  ns  (not significant)

BLEU: F=0.7593, p=0.517717  →  ns  (not significant)

BERTScore: F=1.7151, p=0.163948  →  ns  (not significant)

ONE-WAY ANOVA — metrics by MODEL

SAS: F=29.1436, p=0.000000  →  *** p<0.001
       Multiple Comparison of Means - Tukey HSD, FWER=0.05       
    group1        group2    meandiff p-adj   lower  upper  reject
-----------------------------------------------------------------
 llama-3.1-8b llama-4-scout   0.0387    0.0  0.0251 0.0524   True
 llama-3.1-8b    mistral-7b   0.0011 0.9795 -0.0125 0.0147  False
llama-4-scout    mistral-7b  -0.0376    0.0 -0.0512 -0.024   True
-----------------------------------------------------------------

ROUGE_L: F=116.1994, p=0.000000  →  *** p<0.001
       Multiple Comparison of Means - Tukey HSD, FWER=0.05        
    group1        group2    meandiff p-adj   lower   upper  reject
-------------